# 第9回　データの表現：グラフ・散布図
## ―― グラフを選ぶことは、何を主張するかを選ぶことである

情報活用Ⅰ　／　北星学園大学　2026年度後期

**日本語フォントは、下の準備セルで設定済み。** そのまま日本語のラベルが出る。
（Colabには日本語フォントが入っていないので、何もしないと軸ラベルが □□□ になる）

In [ ]:
# 準備：ライブラリと、練習用データ（北辰大学の学生400人）を読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語が豆腐（□）にならないようにする

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)
except Exception:
    # ネットから取れないときは、同じデータをその場で作る（中身は気にしなくてよい）
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan

print("読み込めた行数:", len(df))
df.head()

---
## 1. グラフの種類と、それが答えている問い

**グラフは飾りではない。問いに対する答えの形である。**

| 問い | グラフ | 使う関数 |
|---|---|---|
| どれが多いか（グループの比較） | 棒グラフ | `plt.bar` |
| どう散らばっているか（1つの項目の分布） | ヒストグラム | `plt.hist` |
| 2つの項目に関係があるか | 散布図 | `plt.scatter` |
| グループごとに関係が違うか | 層別散布図 | `plt.scatter` を色分け |
| グループ間で分布を比べたい | 箱ひげ図 | `plt.boxplot` |

**先に問いを決める。それからグラフを選ぶ。** 逆をやると、きれいだが何も言っていない図ができる。

### 棒グラフ ―― どれが多いか

In [ ]:
counts = df["学部"].value_counts()

plt.figure(figsize=(7, 4))
plt.bar(counts.index, counts.values, color="#80cbc4", edgecolor="white")
plt.ylabel("人数")
plt.title("学部別の人数")
for i, v in enumerate(counts.values):          # 数値を添える
    plt.text(i, v + 3, str(v), ha="center")
plt.show()

### ヒストグラム ―― どう散らばっているか

**平均だけでは、分布の形は分からない。** 山が1つなのか2つなのか、右に裾を引いているのか。

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["睡眠時間h"].dropna(), bins=25, color="#80cbc4", edgecolor="white")
plt.axvline(df["睡眠時間h"].mean(), color="#e8503a", lw=2,
            label=f"平均 {df['睡眠時間h'].mean():.2f}")
plt.axvline(df["睡眠時間h"].median(), color="#1565c0", lw=2, ls="--",
            label=f"中央値 {df['睡眠時間h'].median():.2f}")
plt.xlabel("睡眠時間（時間）"); plt.ylabel("人数")
plt.title("睡眠時間の分布")
plt.legend(); plt.show()

In [ ]:
# 右に裾を引く分布では、平均と中央値が離れる
plt.figure(figsize=(8, 4))
plt.hist(df["世帯年収万円"], bins=40, color="#ffcc80", edgecolor="white")
plt.axvline(df["世帯年収万円"].mean(), color="#e8503a", lw=2,
            label=f"平均 {df['世帯年収万円'].mean():.0f}万円")
plt.axvline(df["世帯年収万円"].median(), color="#1565c0", lw=2, ls="--",
            label=f"中央値 {df['世帯年収万円'].median():.0f}万円")
plt.xlabel("世帯年収（万円）"); plt.ylabel("人数")
plt.title("平均と中央値が100万円近くずれる")
plt.legend(); plt.show()

### 散布図 ―― 2つの項目に関係があるか

**散布図には相関係数を添える。** 目で見た印象と、数字は食い違うことがある。

In [ ]:
# 外れ値（単位ミスの1名）を除いてから描く
d = df[df["身長cm"] < 250].dropna(subset=["睡眠時間h"])

x, y = d["勉強時間h"], d["テスト点"]
r = x.corr(y)

plt.figure(figsize=(6.5, 5))
plt.scatter(x, y, alpha=0.45, s=22, color="#1565c0")
plt.xlabel("勉強時間（時間）"); plt.ylabel("テスト点")
plt.title(f"勉強時間とテスト点　r = {r:.2f}")
plt.show()

print(f"相関係数 r = {r:.3f}")
print("※ 相関があっても、原因と結果は決まらない（第10回で扱う）")

### 層別散布図 ―― グループごとに関係が違うか

In [ ]:
plt.figure(figsize=(7, 5))
colors = {"経済学部": "#1565c0", "文学部": "#e8503a", "社会福祉学部": "#2e7d32"}

for gk, g in d.groupby("学部"):
    rr = g["勉強時間h"].corr(g["テスト点"])
    plt.scatter(g["勉強時間h"], g["テスト点"], alpha=0.5, s=24,
                color=colors[gk], label=f"{gk} (n={len(g)}, r={rr:.2f})")

plt.xlabel("勉強時間（時間）"); plt.ylabel("テスト点")
plt.title("学部で分けて見る")
plt.legend(); plt.show()

**分けて見ると、グループごとに向きや強さが違うことがある。**
第7回のシンプソンのパラドックスと同じ話が、散布図でも起きる。

---
## 2. 悪いグラフ ―― Y軸を0から始めない

**同じデータである。** 描き方だけを変えて2枚並べる。

In [ ]:
means = df.groupby("学部")["テスト点"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))

ax[0].bar(means.index, means.values, color="#80cbc4", edgecolor="white")
ax[0].set_ylim(0, 70)
ax[0].set_ylabel("平均点")
ax[0].set_title("Y軸を0から　── 差はほとんど見えない")

ax[1].bar(means.index, means.values, color="#ef9a9a", edgecolor="white")
ax[1].set_ylim(56, 59)                      # ← ここだけ変えた
ax[1].set_ylabel("平均点")
ax[1].set_title("Y軸を56から　── 圧倒的な差に見える")

plt.tight_layout(); plt.show()

print(f"実際の差（1位と3位）: {means.max() - means.min():.1f} 点")
print(f"個人のばらつき（標準偏差）: {df['テスト点'].std():.1f} 点")

右のグラフは、**嘘をついていない。** 数字は正しく、軸にも目盛りが書いてある。
それでも、読む人は「経済学部が圧勝」という印象を持って帰る。

> **棒グラフのY軸は0から始める。** 棒の長さで量を表す図だから、途中で切ると長さの比が壊れる。
> どうしても拡大したいときは、折れ線にするか、拡大していることを図中に明記する。

---
## 3. 優れた可視化とは何か

悪い例だけを見ると「気をつけよう」で終わる。**何がよい可視化なのか**を2つ挙げる。

### ジョン・スノウのコレラ地図（1854年・ロンドン）

コレラの死亡者を、**ロンドンの地図の上に点で打った。** すると、ブロード街の井戸のまわりに点が集中していることが見えた。井戸が原因だという仮説が、そこから立った。

**グラフの形が、問いの立て方そのものだった例である。** death数を表で並べても、この発見はなかった。

### Our World in Data のグラフ

軸・単位・出典・注釈がすべて明示されており、**元データをダウンロードして自分で確かめられる。**

「確かめられる形で見せる」ことが、可視化の要件でもある。
**これは、AIの出力に対してこの授業がとる態度と同じ話である** ―― 過程が残っているか、確かめられるか。

> あなたが自分のページに図を載せるときも（第10・11回）、> **出典と、何を数えた数字なのかを必ず添える。**

---
## 4. こういう見せ方もある（紹介）

In [ ]:
# 箱ひげ図：グループごとの分布を並べて比べる
groups = [g["テスト点"].values for _, g in df.groupby("学部")]
labels = [gk for gk, _ in df.groupby("学部")]

plt.figure(figsize=(7, 4.5))
plt.boxplot(groups, patch_artist=True,
            boxprops=dict(facecolor="#b2dfdb"), medianprops=dict(color="#e8503a", lw=2))
plt.xticks(range(1, len(labels) + 1), labels)   # 目盛りラベルは別に指定する
plt.ylabel("テスト点")
plt.title("学部別テスト点の分布（箱＝真ん中の50%、線＝中央値）")
plt.show()

In [ ]:
# ヒートマップ：数値どうしの関係を一覧する
cols = ["睡眠時間h", "SNS時間h", "勉強時間h", "アルバイト時間week", "出席率", "テスト点"]
corr = df[cols].corr()

plt.figure(figsize=(6.5, 5.5))
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="相関係数")
plt.xticks(range(len(cols)), cols, rotation=45, ha="right")
plt.yticks(range(len(cols)), cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.title("項目どうしの相関")
plt.tight_layout(); plt.show()

---
## 5. 余談 ―― あなたはすでにコマンドを使っている

Colabのセルの先頭に `!` をつけると、**コンピュータに直接命令できる。**

In [ ]:
!ls

いま出てきたのは、**このノートブックが動いている場所にあるファイルの一覧**である。

普段は、フォルダのアイコンをダブルクリックして中身を見る。あれと同じことを、文字で命令した。
前者を **GUI**（画面で操作する）、後者を **CLI**（文字で命令する）という。

第6回で `pd.read_csv("ファイル名")` と書いたとき、あなたはすでに**「どこに何というファイルがあるか」を文字で指定していた。** CLIの世界に片足を入れている。

画面の裏側には、必ずファイルの実体がある。**アイコンは、その見え方の一つでしかない。**

---
## 6. 画像に書き出す（次回、ページに貼る）

第10回で、成果物のページにこのグラフを貼る。**いま画像として書き出しておく。**

In [ ]:
plt.figure(figsize=(8, 5))
for gk, g in d.groupby("学部"):
    plt.scatter(g["勉強時間h"], g["テスト点"], alpha=0.5, s=24, label=gk)
plt.xlabel("勉強時間（時間）"); plt.ylabel("テスト点")
plt.title("勉強時間とテスト点（学部別）")
plt.legend()

# ← これが書き出しの行。dpi=150 くらいにするとページで粗く見えない
plt.savefig("graph1.png", dpi=150, bbox_inches="tight")
plt.show()

print("graph1.png を書き出した。左の📁からダウンロードできる。")

---
## 7. 自分の調査データでやる

In [ ]:
# ここから先は「自分の調査データ」でやる。
# Google Forms の回答 → スプレッドシート → ファイル → ダウンロード → CSV で書き出したものを使う。
#
# 左のフォルダアイコン（📁）にCSVをドラッグしてから、ファイル名を書き換えて実行する。
# ※ 数字を手で打ち直さないこと。転記した瞬間に、それは元データではなくなる。

# mydf = pd.read_csv("自分のファイル名.csv")
# mydf.head()

In [ ]:
# グラフ1枚目（列名を書き換える）
# plt.figure(figsize=(7, 4))
# plt.hist(mydf["数値の列"].dropna(), bins=15, color="#80cbc4", edgecolor="white")
# plt.xlabel("　"); plt.ylabel("人数"); plt.title("　")
# plt.savefig("mygraph1.png", dpi=150, bbox_inches="tight")
# plt.show()

---
## 課題9（8点）

**このノートブック** ＋ **グラフ2枚（うち1枚は層別）** ＋ **各グラフの読み取り**。

- [ ] グラフ1枚目。軸ラベルとタイトルを必ず入れる
- [ ] グラフ2枚目は**層別**（何かで色分け・グループ分けする）
- [ ] 散布図を描いたなら、**相関係数を添える**
- [ ] 各グラフについて、読み取ったことを1〜2文
- [ ] **画像を書き出す**（`savefig`）。次回のページで使う

**軸ラベルのないグラフは減点する。** 何の数字かが分からない図は、根拠にならない。

提出期限：次回授業の開始まで（遅れた場合は50%）